# Inkling 2 Mini → block diffusion on one A100 80GB

This notebook drives the `inkling_dlm` package (https://github.com/NarendraPatwardhan/inkling-dlm). One flow:

1. **Stage 1 — GPU gates** (≈ 15–25 min, a few CU): load (BF16 dequant, text path), dispatch (`grouped_mm`), LoRA-off
   logit equivalence, read-only cache, gradient flow, gradients through the cache under checkpointing, memory and
   speed with the **projected compute-unit cost**. Any failure — including a projection above 200 CU — stops here.
2. **Stage 2 — training** (25,000 updates). Runs **only** when `TRAIN_APPROVED = True` (the owner's approval).
   Detached; checkpoints every 500 updates to Drive; re-running the notebook resumes.
3. **Stage 3 — sanity checks** (held-out masked CE, copy test, code samples) and **stop the runtime**.

Everything is logged to W&B (`narendra-patwardhan-independent/inkling-dlm`). The key is read from a Colab secret
(`WANDB_API_KEY`) or `/content/wandb.key` and is never printed. Runtime: **A100, 80 GB** (the 40 GB A100 cannot hold
the 52 GB BF16 model).

In [ ]:
# Parameters
REPO = "https://github.com/NarendraPatwardhan/inkling-dlm.git"
COMMIT = "main"                  # pin a commit SHA for the real run
CONFIG = "configs/a100.yaml"
TRAIN_APPROVED = False           # set True only after the owner has approved the main training run
RUN_DIR = "/content/drive/MyDrive/inkling-dlm/runs/main"
J = "/content/inkling-dlm"

In [ ]:
# Hardware: the BF16 model is ~52 GB, so a 40 GB A100 cannot run this
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(gpu)
name, mem = gpu.rsplit(",", 1)
assert "A100" in name and int(mem) > 70_000, f"need an A100 80GB runtime, got {gpu}"


In [ ]:
# Install the package at the pinned commit (keeps Colab's torch; pins transformers/peft/accelerate)
import os
if not os.path.exists(J):
    !git clone -q {REPO} {J}
!cd {J} && git fetch -q origin && git checkout -q {COMMIT} && git log -1 --format='%H %s'
!pip install -q -e {J}
!python -c "import torch, transformers, peft; print(torch.__version__, transformers.__version__, peft.__version__)"


In [ ]:
# Google Drive: checkpoints, data cache, gates.json, sanity.json
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUN_DIR, exist_ok=True)


In [ ]:
# W&B key -> /content/wandb.key (chmod 600). Never printed.
import pathlib, getpass
key_file = pathlib.Path("/content/wandb.key")
if not key_file.exists():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        pass
    key = key or getpass.getpass("W&B API key (hidden): ")
    key_file.write_text(key.strip()); key_file.chmod(0o600); del key
os.environ["WANDB_KEY_FILE"] = str(key_file)
print("W&B key: available" if key_file.stat().st_size > 0 else "W&B key: MISSING")


## Stage 1 — GPU gates (must all pass)

In [ ]:
!cd {J} && python -m inkling_dlm gates --config {CONFIG}
import json
g = json.load(open(f"{RUN_DIR}/gates.json"))
for r in g["gates"]:
    print(("PASS " if r["passed"] else "FAIL ") + r["gate"].ljust(24) + r["detail"])
print("setting:", g["setting"]); print("projection:", g["projection"])
assert g["passed"], "a gate failed: do not train"


## Stage 2 — training (only with the owner's approval)

In [ ]:
if not TRAIN_APPROVED:
    raise SystemExit("Gates passed. Training needs the owner's approval: set TRAIN_APPROVED = True and re-run.")
import subprocess
log = f"{RUN_DIR}/train.log"
cmd = f"cd {J} && nohup python -m inkling_dlm train --config {CONFIG} --approved >> {log} 2>&1 &"
subprocess.run(["bash", "-c", cmd], env=dict(os.environ, PYTHONUNBUFFERED="1"))
print("training launched (detached); it resumes from the newest Drive checkpoint if one exists")


In [ ]:
# Monitor (re-run any time): last log lines and the newest checkpoint
import glob
print("".join(open(f"{RUN_DIR}/train.log").readlines()[-15:]))
print(sorted(p for p in glob.glob(f"{RUN_DIR}/state-*") if not p.endswith(".partial"))[-1:])


## Stage 3 — sanity checks, then stop the runtime

In [ ]:
!cd {J} && python -m inkling_dlm sanity --config {CONFIG}
s = json.load(open(f"{RUN_DIR}/sanity.json"))
print("held-out CE  parent:", round(s["init"]["heldout"]["ce"], 3), " trained:", round(s["final"]["heldout"]["ce"], 3))
print("copy test    parent:", s["init"]["copy"], " trained:", s["final"]["copy"])
print("parse rates:", s["final"]["parse_rate"])


In [ ]:
# Never leave the runtime idle
from google.colab import runtime
runtime.unassign()
